# 04 – Splits, Cross Validation und Leakage
Nicht jeder Datensatz darf zufällig gemischt werden. Wir vergleichen stratified, group- und zeitliche Splits und zeigen, wie die Pipeline Vorverarbeitung innerhalb jedes Folds neu lernt.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold, TimeSeriesSplit, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

## Stratified Split bei seltenen Klassen

In [ ]:
X,y=make_classification(n_samples=500,n_features=6,weights=[.92,.08],random_state=7)
_,_,y_train_a,y_test_a=train_test_split(X,y,test_size=.2,random_state=3)
_,_,y_train_b,y_test_b=train_test_split(X,y,test_size=.2,random_state=3,stratify=y)
vergleich=pd.DataFrame({"gesamt":pd.Series(y).value_counts(normalize=True),"ohne stratify - train":pd.Series(y_train_a).value_counts(normalize=True),"ohne stratify - test":pd.Series(y_test_a).value_counts(normalize=True),"mit stratify - train":pd.Series(y_train_b).value_counts(normalize=True),"mit stratify - test":pd.Series(y_test_b).value_counts(normalize=True)}).mul(100).round(1)
display(vergleich)

## Pipeline in der Cross Validation

In [ ]:
pipeline=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000))
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
ergebnis=cross_validate(pipeline,X,y,cv=cv,scoring=["accuracy","balanced_accuracy","f1"],return_estimator=True)
display(pd.DataFrame({k:v for k,v in ergebnis.items() if k.startswith("test_")}).round(3))
scaler_mittelwerte=np.vstack([e.named_steps["standardscaler"].mean_ for e in ergebnis["estimator"]])
display(pd.DataFrame(scaler_mittelwerte,columns=[f"Feature {i}" for i in range(X.shape[1])]).round(3))

Die Mittelwerte unterscheiden sich leicht: Jeder Scaler wurde nur auf dem Trainingsanteil seines Folds gelernt. Genau das verhindert Leakage.

## Group Split
Mehrere Messungen derselben Person oder Maschine müssen im selben Fold bleiben.

In [ ]:
gruppen=np.repeat(np.arange(30),4); X_gruppe=np.column_stack([gruppen,np.arange(len(gruppen))]); y_gruppe=gruppen%2
gkf=GroupKFold(n_splits=5); zeilen=[]
for fold,(train_idx,test_idx) in enumerate(gkf.split(X_gruppe,y_gruppe,groups=gruppen),1):
    train_groups=set(gruppen[train_idx]); test_groups=set(gruppen[test_idx]); zeilen.append({"Fold":fold,"Train-Samples":len(train_idx),"Test-Samples":len(test_idx),"gemeinsame Gruppen":len(train_groups & test_groups)})
display(pd.DataFrame(zeilen))

## Zeitreihen-Split
Training liegt zeitlich immer vor dem Test.

In [ ]:
tscv=TimeSeriesSplit(n_splits=5); fig,ax=plt.subplots(figsize=(10,4))
for fold,(train_idx,test_idx) in enumerate(tscv.split(np.arange(120))):
    ax.scatter(train_idx,np.full_like(train_idx,fold),marker='s',s=12,color="steelblue")
    ax.scatter(test_idx,np.full_like(test_idx,fold),marker='s',s=12,color="firebrick")
ax.set(title="TimeSeriesSplit: Blau = Training, Rot = Test",xlabel="Zeitindex",ylabel="Fold"); plt.show()

Die Split-Strategie folgt der späteren Anwendung: neue Samples, neue Gruppen oder zukünftige Zeitpunkte. Ein zufälliger Standardsplit ist nicht automatisch realistisch.